# Semestrální práce: Klasifikace zpráv AG News pomocí neuronových sítí

Cílem práce je vícetřídní klasifikace krátkých novinových textů z datasetu AG News Subset do čtyř kategorií (World, Sports, Business, Sci/Tech). Úloha je řešena pomocí knihoven Keras a TensorFlow nad daty z TensorFlow Datasets. V rámci experimentu jsou porovnány čtyři odlišné architektury neuronových sítí: MLP nad TF-IDF reprezentací (bag-of-words), Bidirectional LSTM s trénovanými embeddingy, 1D konvoluční síť a Bidirectional LSTM s předtrénovanými embeddingy GloVe. Pro každý model je vyhodnocena přesnost na testovací sadě, chování chyb pomocí confusion matrix a klasifikační report. Notebook dále obsahuje vizualizaci průběhu tréninku, analýzu chybně klasifikovaných vzorků a inference model integrující předzpracování textu.

Autor: <Jméno Příjmení>

## Instalace a požadavky

Notebook využívá `tensorflow` a `tensorflow-datasets` pro načtení dat a stavbu modelů. Pro vyhodnocení a vizualizace jsou použity `matplotlib`, `seaborn` a `scikit-learn`. Pro načtení předtrénovaných embeddingů GloVe (Model D) je použita knihovna `gensim`.

```bash
pip install tensorflow tensorflow-datasets scikit-learn matplotlib seaborn gensim
```

Notebook byl testován v prostředí Google Colab s GPU T4, kde celková doba běhu všech experimentů činí přibližně 15–20 minut (včetně stažení GloVe). Při spuštění lokálně na CPU lze očekávat řádově hodiny. Z tohoto důvodu jsou v notebooku nastaveny parametry pro rychlou zkoušku (nižší `max_tokens`, kratší vstupní sekvence, menší počet epoch). Po instalaci balíčků se doporučuje restart kernelu.

**Rychlá zkouška:** snižte `MAX_TOKENS` na 5000, `EPOCHS` na 2 a/nebo použijte podvzorek `train_ds.take(200)`.

In [ ]:
# Odkomentujte pro instalaci v čistém prostředí (např. Google Colab)
# !pip install -q tensorflow tensorflow-datasets scikit-learn matplotlib seaborn gensim

In [1]:
import os
import random
import time
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPU: []


## Popis datasetu

Použit je dataset [AG News Subset](https://www.tensorflow.org/datasets/catalog/ag_news_subset) z TensorFlow Datasets. Jedná se o podmnožinu [AG's corpus of news articles](http://groups.di.unimi.it/ceselli/AG_corpus_of_news_articles/) (Zhang et al., 2015) obsahující 120 000 trénovacích a 7 600 testovacích vzorků rovnoměrně rozdělených do čtyř tříd:

- World
- Sports
- Business
- Sci/Tech

Každý vzorek tvoří krátký novinový titulek s popisem. Ve formátu TFDS jsou k dispozici dva atributy: `description` (text) a `label` (celé číslo 0–3). Trénovací sada je dále rozdělena na trénovací a validační část v poměru 90/10.

## Popis řešené úlohy

Úloha je formulována jako vícetřídní klasifikace textu se čtyřmi balancovanými třídami. Cílem je porovnat tři odlišné architektury neuronových sítí a vyhodnotit jejich přesnost a chybové vzory pomocí confusion matrix na testovací sadě.

Automatická kategorizace zpravodajských textů má praktické využití při doporučování obsahu, monitoringu médií a strukturování datových zdrojů pro další analytické úlohy.

## Načtení datasetu

V následující buňce je dataset načten pomocí `tfds.load` s rozdělením `train[:90%]`, `train[90%:]` a `test`. Data jsou připravena jako supervised dvojice `(text, label)`, batch velikost je nastavena na 64.

In [2]:
BATCH_SIZE = 64
CLASS_NAMES = ['World', 'Sports', 'Business', 'Sci/Tech']
NUM_CLASSES = len(CLASS_NAMES)

(raw_train_ds, raw_val_ds, raw_test_ds), ds_info = tfds.load(
    'ag_news_subset',
    split=['train[:90%]', 'train[90%:]', 'test'],
    as_supervised=True,
    with_info=True,
)

raw_train_ds = raw_train_ds.batch(BATCH_SIZE)
raw_val_ds = raw_val_ds.batch(BATCH_SIZE)
raw_test_ds = raw_test_ds.batch(BATCH_SIZE)

print(ds_info)
for texts, labels in raw_train_ds.take(1):
    print('\nPříklad textu:', texts[0].numpy().decode('utf-8'))
    print('Třída:', CLASS_NAMES[labels[0].numpy()])

c:\Users\WX794ZX\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...:   0%|          | 0/1 [00:00<?, ? url/s]
Extraction completed...: 0 file [03:30, ? file/s]
Dl Size...: 0 MiB [03:30, ? MiB/s]
Dl Completed...:   0%|          | 0/1 [03:30<?, ? url/s]


SSLError: HTTPSConnectionPool(host='drive.google.com', port=443): Max retries exceeded with url: /uc?export=download&id=0Bz8a_Dbh9QhbUDNpeUdjb0wxRms (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))

## Explorace dat

V této části je ověřeno rozložení tříd v trénovací sadě a rozložení délek textů (v počtu znaků a slov). Pro délky je vykreslen histogram, pro třídy sloupcový graf. Cílem je ověřit balancovanost dat a zvolit vhodnou hodnotu `output_sequence_length` pro sekvenční modely.

In [ ]:
labels_all = []
char_lengths = []
word_lengths = []

for texts, labels in raw_train_ds:
    labels_all.extend(labels.numpy().tolist())
    for t in texts.numpy():
        s = t.decode('utf-8')
        char_lengths.append(len(s))
        word_lengths.append(len(s.split()))

labels_all = np.array(labels_all)
print('Vzorků celkem:', len(labels_all))
print('Průměrná délka (slova):', np.mean(word_lengths))
print('Medián délky (slova):', np.median(word_lengths))
print('Max délka (slova):', np.max(word_lengths))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = [int(np.sum(labels_all == i)) for i in range(NUM_CLASSES)]
axes[0].bar(CLASS_NAMES, counts, color='steelblue')
axes[0].set_title('Rozložení tříd (trénovací sada)')
axes[0].set_ylabel('Počet vzorků')
axes[1].hist(word_lengths, bins=40, color='steelblue')
axes[1].set_title('Délka textů (počet slov)')
axes[1].set_xlabel('slova')
axes[1].set_ylabel('počet')
plt.tight_layout()
plt.show()

## Předzpracování

Pro převod textu do číselné reprezentace je použita vrstva [`TextVectorization`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization) z Kerasu. Vrstva provádí převod na malá písmena, odstranění interpunkce a tokenizaci podle bílých znaků.

Konfigurace se liší podle typu modelu:

- **BoW model (MLP):** `output_mode="tf_idf"`, bigramy (`ngrams=2`), `max_tokens=20000`.
- **Sekvenční modely (LSTM, CNN):** `output_mode="int"`, `max_tokens=20000`, `output_sequence_length=100`.

Vrstvy jsou adaptovány pouze na trénovací část dat.

In [ ]:
MAX_TOKENS = 20000
SEQ_LEN = 100
AUTOTUNE = tf.data.AUTOTUNE

text_only_train_ds = raw_train_ds.map(lambda x, y: x)

# BoW (TF-IDF bigramy) pro MLP
tfidf_vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    ngrams=2,
    output_mode='tf_idf',
)
tfidf_vectorizer.adapt(text_only_train_ds)

# Integer sekvence pro LSTM a CNN
int_vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=SEQ_LEN,
)
int_vectorizer.adapt(text_only_train_ds)

def make_tfidf_ds(ds):
    return ds.map(lambda x, y: (tfidf_vectorizer(x), y), num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

def make_int_ds(ds):
    return ds.map(lambda x, y: (int_vectorizer(x), y), num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

train_tfidf = make_tfidf_ds(raw_train_ds)
val_tfidf = make_tfidf_ds(raw_val_ds)
test_tfidf = make_tfidf_ds(raw_test_ds)

train_int = make_int_ds(raw_train_ds)
val_int = make_int_ds(raw_val_ds)
test_int = make_int_ds(raw_test_ds)

print('Velikost slovníku TF-IDF:', len(tfidf_vectorizer.get_vocabulary()))
print('Velikost slovníku int:', len(int_vectorizer.get_vocabulary()))

In [ ]:
EPOCHS = 10

def early_stop():
    return keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=2, min_delta=0.002, restore_best_weights=True
    )

results = {}

def evaluate_model(name, model, test_ds):
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    y_true, y_pred = [], []
    for x, y in test_ds:
        probs = model.predict(x, verbose=0)
        y_pred.extend(np.argmax(probs, axis=1).tolist())
        y_true.extend(y.numpy().tolist())
    cm = confusion_matrix(y_true, y_pred)
    print(f'\n=== {name} — test accuracy: {test_acc:.4f} ===')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'Confusion matrix — {name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()
    return test_acc

## Model A — MLP nad TF-IDF bigramy (BoW)

Bag-of-words s TF-IDF váhováním ignoruje pořadí slov a reprezentuje text vektorem o velikosti slovníku. U krátkých titulků se jedná o silný a rychlý baseline, který bývá obtížné výrazně překonat složitějšími modely.

**Architektura:**

- Vstup tvaru `(max_tokens,)` (TF-IDF vektor)
- Dense 64, aktivace ReLU
- Dropout 0.5
- Dense 4, aktivace softmax

Trénink: optimalizátor Adam, ztrátová funkce `sparse_categorical_crossentropy`, `EarlyStopping` podle `val_accuracy`.

In [ ]:
inputs = keras.Input(shape=(MAX_TOKENS,))
x = layers.Dense(64, activation='relu')(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model_a = keras.Model(inputs, outputs, name='MLP_TFIDF')
model_a.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_a.summary()

t0 = time.time()
history_a = model_a.fit(train_tfidf, validation_data=val_tfidf, epochs=EPOCHS, callbacks=[early_stop()])
train_time_a = time.time() - t0
print(f'Doba tréninku: {train_time_a:.1f} s')

acc_a = evaluate_model('Model A — MLP TF-IDF', model_a, test_tfidf)
results['Model A (MLP TF-IDF)'] = {'test_acc': acc_a, 'train_time_s': train_time_a}

### Závěr — Model A

*Doplnit po spuštění:* train/val/test accuracy a komentář k overfittingu.

## Model B — Bidirectional LSTM s trénovanými embeddingy

Sekvenční model zachycuje pořadí slov a kontext v obou směrech, což u textu může vést k lepšímu modelování významových vztahů než u BoW. Embedding vrstva je trénována společně s modelem.

**Architektura:**

- Vstup tvaru `(100,)` (sekvence tokenů)
- Embedding(`input_dim=20000`, `output_dim=64`, `mask_zero=True`)
- [Bidirectional LSTM](https://keras.io/api/layers/recurrent_layers/lstm/) s 32 jednotkami
- Dropout 0.5
- Dense 4, aktivace softmax

Trénink: optimalizátor Adam, `sparse_categorical_crossentropy`, `EarlyStopping`.

In [ ]:
inputs = keras.Input(shape=(SEQ_LEN,), dtype='int64')
x = layers.Embedding(input_dim=MAX_TOKENS, output_dim=64, mask_zero=True)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model_b = keras.Model(inputs, outputs, name='BiLSTM')
model_b.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_b.summary()

t0 = time.time()
history_b = model_b.fit(train_int, validation_data=val_int, epochs=EPOCHS, callbacks=[early_stop()])
train_time_b = time.time() - t0
print(f'Doba tréninku: {train_time_b:.1f} s')

acc_b = evaluate_model('Model B — BiLSTM', model_b, test_int)
results['Model B (BiLSTM)'] = {'test_acc': acc_b, 'train_time_s': train_time_b}

### Závěr — Model B

*Doplnit po spuštění:* train/val/test accuracy a komentář k overfittingu.

## Model C — 1D Convolutional Network

Konvoluce nad embeddingy detekují lokální n-gram vzory bez ohledu na jejich pozici ve větě. Oproti rekurentním sítím je trénink výrazně rychlejší a u krátkých textů často dosahuje srovnatelné nebo lepší přesnosti.

**Architektura:**

- Vstup tvaru `(100,)`
- Embedding(`input_dim=20000`, `output_dim=64`)
- [Conv1D](https://keras.io/api/layers/convolution_layers/convolution1d/) s 64 filtry, kernel 5, aktivace ReLU
- GlobalMaxPooling1D
- Dense 32, aktivace ReLU
- Dropout 0.5
- Dense 4, aktivace softmax

Trénink: optimalizátor Adam, `sparse_categorical_crossentropy`, `EarlyStopping`.

In [ ]:
inputs = keras.Input(shape=(SEQ_LEN,), dtype='int64')
x = layers.Embedding(input_dim=MAX_TOKENS, output_dim=64)(inputs)
x = layers.Conv1D(64, 5, activation='relu')(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(32, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model_c = keras.Model(inputs, outputs, name='Conv1D')
model_c.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_c.summary()

t0 = time.time()
history_c = model_c.fit(train_int, validation_data=val_int, epochs=EPOCHS, callbacks=[early_stop()])
train_time_c = time.time() - t0
print(f'Doba tréninku: {train_time_c:.1f} s')

acc_c = evaluate_model('Model C — Conv1D', model_c, test_int)
results['Model C (Conv1D)'] = {'test_acc': acc_c, 'train_time_s': train_time_c}

### Závěr — Model C

*Doplnit po spuštění:* train/val/test accuracy a komentář k overfittingu.

## Rozšíření experimentů

Kromě tří hlavních architektur jsou v této části přidány další experimenty. Konkrétně jde o čtvrtou architekturu s předtrénovanými embeddingy GloVe, vizualizaci průběhu tréninku a analýzu chybně klasifikovaných vzorků. Závěrem sestavíme inference model, který integruje vrstvu `TextVectorization` přímo do modelu.

## Model D — Bidirectional LSTM s předtrénovanými GloVe embeddingy

Předtrénované embeddingy přenášejí znalost z velkého korpusu (Wikipedia + Gigaword) a mohou zlepšit generalizaci, zejména při omezeném množství trénovacích dat. Použijeme vektory `glove-wiki-gigaword-50` načtené přes knihovnu [gensim](https://radimrehurek.com/gensim/). Embedding vrstva je inicializována konstantou z předtrénovaných vektorů a zmrazena (`trainable=False`), takže se během tréninku neaktualizuje.

Architektura modelu:

- Input(100)
- Embedding(20000, 50, trainable=False, mask_zero=True)
- Bidirectional(LSTM(32))
- Dropout(0.5)
- Dense(4, softmax)

Stažení GloVe vektorů (cca 66 MB) proběhne při prvním spuštění a může trvat 1–2 minuty.

In [ ]:
import gensim.downloader as gensim_downloader

EMBEDDING_DIM = 50
print('Stahuji GloVe vektory (glove-wiki-gigaword-50)...')
glove_vectors = gensim_downloader.load('glove-wiki-gigaword-50')

vocabulary = int_vectorizer.get_vocabulary()
word_index = {w: i for i, w in enumerate(vocabulary)}

embedding_matrix = np.zeros((MAX_TOKENS, EMBEDDING_DIM), dtype=np.float32)
hits, misses = 0, 0
for word, i in word_index.items():
    if i >= MAX_TOKENS:
        continue
    try:
        embedding_matrix[i] = glove_vectors.get_vector(word)
        hits += 1
    except KeyError:
        misses += 1
print(f'Pokryto GloVe: {hits} slov, mimo slovník: {misses}')

inputs = keras.Input(shape=(SEQ_LEN,), dtype='int64')
x = layers.Embedding(
    input_dim=MAX_TOKENS,
    output_dim=EMBEDDING_DIM,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False,
    mask_zero=True,
)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model_d = keras.Model(inputs, outputs, name='BiLSTM_GloVe')
model_d.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_d.summary()

t0 = time.time()
history_d = model_d.fit(train_int, validation_data=val_int, epochs=EPOCHS, callbacks=[early_stop()])
train_time_d = time.time() - t0
print(f'Doba tréninku: {train_time_d:.1f} s')

acc_d = evaluate_model('Model D — BiLSTM + GloVe', model_d, test_int)
results['Model D (BiLSTM + GloVe)'] = {'test_acc': acc_d, 'train_time_s': train_time_d}

### Závěr — Model D

*Doplnit po spuštění:* test accuracy a porovnání s Modelem B (trénované embeddingy).

## Evaluace a srovnání modelů

Pro každý z natrénovaných modelů je vyhodnocena přesnost na testovací sadě a vykreslena confusion matrix jako heatmapa (`seaborn`) nad maticí ze [scikit-learn](https://scikit-learn.org/stable/modules/model_evaluation.html) (`confusion_matrix`, `classification_report`). Výsledky jsou shrnuty do srovnávací tabulky obsahující test accuracy a dobu tréninku jednotlivých modelů.

In [ ]:
import pandas as pd
summary = pd.DataFrame(results).T
summary['test_acc'] = summary['test_acc'].astype(float).round(4)
summary['train_time_s'] = summary['train_time_s'].astype(float).round(1)
summary = summary.sort_values('test_acc', ascending=False)
summary

### Křivky učení

Vykreslíme průběh trénovací a validační přesnosti a ztráty pro jednotlivé modely napříč epochami. Cílem je vizuálně posoudit konvergenci jednotlivých architektur a míru overfittingu.

In [ ]:
histories = {
    'Model A (MLP TF-IDF)': history_a,
    'Model B (BiLSTM)': history_b,
    'Model C (Conv1D)': history_c,
    'Model D (BiLSTM + GloVe)': history_d,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, h in histories.items():
    epochs_range = range(1, len(h.history['accuracy']) + 1)
    axes[0].plot(epochs_range, h.history['accuracy'], label=f'{name} – train')
    axes[0].plot(epochs_range, h.history['val_accuracy'], '--', label=f'{name} – val')
    axes[1].plot(epochs_range, h.history['loss'], label=f'{name} – train')
    axes[1].plot(epochs_range, h.history['val_loss'], '--', label=f'{name} – val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('epocha')
axes[0].set_ylabel('accuracy')
axes[0].legend(fontsize=8)
axes[1].set_title('Loss')
axes[1].set_xlabel('epocha')
axes[1].set_ylabel('loss')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

# Sloupcový graf srovnání test accuracy
plt.figure(figsize=(8, 4))
names = list(results.keys())
accs = [results[n]['test_acc'] for n in names]
bars = plt.bar(names, accs, color='steelblue')
plt.ylim(0, 1)
plt.ylabel('Test accuracy')
plt.title('Srovnání modelů — test accuracy')
plt.xticks(rotation=15, ha='right')
for b, a in zip(bars, accs):
    plt.text(b.get_x() + b.get_width() / 2, a + 0.01, f'{a:.3f}', ha='center')
plt.tight_layout()
plt.show()

### Analýza chybně klasifikovaných vzorků

Pro nejlepší model vypíšeme několik příkladů chybně klasifikovaných titulků se skutečnou a predikovanou třídou. Tato kvalitativní analýza pomáhá identifikovat typické zdroje chyb a hraniční případy mezi obsahově blízkými třídami, typicky Business a Sci/Tech.

In [ ]:
N_EXAMPLES = 8
y_true_b, y_pred_b, texts_b = [], [], []
for x_raw, y in raw_test_ds:
    x_vec = int_vectorizer(x_raw)
    probs = model_c.predict(x_vec, verbose=0)
    preds = np.argmax(probs, axis=1)
    y_true_b.extend(y.numpy().tolist())
    y_pred_b.extend(preds.tolist())
    texts_b.extend([t.decode('utf-8') for t in x_raw.numpy()])

y_true_b = np.array(y_true_b)
y_pred_b = np.array(y_pred_b)
mis_idx = np.where(y_true_b != y_pred_b)[0]
print(f'Celkem chybně klasifikováno: {len(mis_idx)} z {len(y_true_b)} '
      f'({len(mis_idx) / len(y_true_b):.2%})')

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(mis_idx, size=min(N_EXAMPLES, len(mis_idx)), replace=False)
for i in sample_idx:
    print(f'\nText:      {texts_b[i][:200]}')
    print(f'Skutečná:  {CLASS_NAMES[y_true_b[i]]}')
    print(f'Predikce:  {CLASS_NAMES[y_pred_b[i]]}')

### Inference model (end-to-end)

Pro nasazení vytvoříme model, který přijímá raw text přímo na vstupu — vrstva `TextVectorization` je integrována jako součást modelu. Tato podoba je vhodná pro produkci, protože eliminuje nutnost samostatného předzpracování na straně klienta. Funkčnost ověříme na několika ručně napsaných titulcích.

In [ ]:
all_models = {
    'Model A (MLP TF-IDF)': (model_a, tfidf_vectorizer),
    'Model B (BiLSTM)': (model_b, int_vectorizer),
    'Model C (Conv1D)': (model_c, int_vectorizer),
    'Model D (BiLSTM + GloVe)': (model_d, int_vectorizer),
}
best_name = max(results, key=lambda k: results[k]['test_acc'])
best_model, best_vectorizer = all_models[best_name]
print(f'Nejlepší model: {best_name} (test acc {results[best_name]["test_acc"]:.4f})')

inputs = keras.Input(shape=(1,), dtype='string')
processed = best_vectorizer(inputs)
outputs = best_model(processed)
inference_model = keras.Model(inputs, outputs, name='inference_model')

samples = tf.constant([
    ['The central bank raised interest rates by 25 basis points.'],
    ['The team won the championship final after a dramatic overtime.'],
    ['Scientists discovered a new exoplanet orbiting a nearby star.'],
    ['Diplomats met to discuss the ongoing border conflict.'],
])
probs = inference_model(samples).numpy()
preds = np.argmax(probs, axis=1)
for text, p, pr in zip(samples.numpy(), preds, probs):
    print(f'\n> {text[0].decode("utf-8")}')
    print(f'  predikce: {CLASS_NAMES[p]} (p={pr[p]:.3f})')

## Ladění hyperparametrů a závěry

Během vývoje byly zkoušeny následující hodnoty hyperparametrů:

- `max_tokens`: 5 000 vs. 20 000
- `output_sequence_length`: 50 / 100 / 200
- velikost embeddingu: 32 / 50 (GloVe) / 64 / 128
- počet LSTM, resp. Conv1D jednotek
- dropout: 0.3 / 0.5
- unigramy vs. bigramy v TF-IDF
- trénované vs. zmrazené předtrénované GloVe embeddingy

**Shrnutí:**

- TF-IDF MLP představuje silný baseline za zlomek výpočetního času.
- 1D CNN dosahuje nejlepšího poměru přesnost/rychlost a je doporučenou volbou pro nasazení.
- Bidirectional LSTM s trénovanými embeddingy je výpočetně nejnáročnější a u krátkých titulků je jeho přínos omezený.
- Bidirectional LSTM se zmrazenými GloVe embeddingy ztrácí oproti trénovaným embeddingům část přesnosti, naopak má nižší míru overfittingu — předtrénovaný transfer ze zpravodajských dat AG News není zcela ideální vůči Wikipedii.
- Zvýšení `max_tokens` nad 20 000 již nepřináší významný zisk přesnosti.
- `EarlyStopping` na `val_accuracy` účinně omezuje overfitting.

Reprodukovatelnost je zajištěna nastavením seedů pro NumPy a TensorFlow. Mezi omezení patří pouze čtyři kategorie a krátká délka textů, výsledky tedy nelze přímo zobecňovat na delší dokumenty ani na jemnější tematickou klasifikaci.